# NB03b — Retrieval diagnosis & one-round hybrid improvement

**Goal:** Diagnose why the current legal-aware Top-5 retrieval failed on some of Q1–Q6, then test one general retrieval improvement without changing the benchmark questions or the legal corpus.

This notebook does **not** overwrite the official NB03 result. It creates a candidate retrieval result for review.

### Important experimental rule

The benchmark-specific support rules below are used **only for evaluation/diagnosis**.  
They are **never used to rank or boost chunks**.

That distinction matters: otherwise we would simply tune the retriever to the known answers.

### What this notebook tests

1. **Library coverage:** Is the needed legal evidence actually present in `chunks_legal_aware.json`?
2. **Current baseline:** Dense embedding + cosine similarity, Top-5.
3. **Candidate v2:** Hybrid retrieval:
   - dense semantic ranking
   - BM25 lexical ranking
   - Reciprocal Rank Fusion (RRF)
   - light source diversification
4. Compare evidence coverage at Top-5 and Top-10 for Q1–Q6.


In [1]:
# Run once if needed
# %pip install -q -U sentence-transformers rank-bm25 pandas numpy


In [2]:
from pathlib import Path
import json, re, hashlib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from IPython.display import display

DATA_DIR = Path("data")
LEGAL_PATH = DATA_DIR / "chunks_legal_aware.json"
BENCHMARK_PATH = DATA_DIR / "benchmark_questions_v1.json"

for p in [LEGAL_PATH, BENCHMARK_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p}")

legal_chunks = json.loads(LEGAL_PATH.read_text(encoding="utf-8"))
benchmark = json.loads(BENCHMARK_PATH.read_text(encoding="utf-8"))
questions = benchmark["questions"]

assert list(questions) == [f"Q{i}" for i in range(1, 11)]
BENCHMARK_SHA256 = hashlib.sha256(
    json.dumps(questions, ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest()

print("Benchmark:", benchmark["benchmark_id"])
print("Legal-aware chunks:", len(legal_chunks))
print("Benchmark SHA256:", BENCHMARK_SHA256)


c:\Users\SAMSUNG\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Benchmark: regulatory_agent_v1_notion_2026-10-08
Legal-aware chunks: 1383
Benchmark SHA256: 2ee3bf32133c31bc62514cfa85e116bd5dd846a89b282a1a6f62f1ec849c3e4e


## 1. Build the searchable text

We keep the **same legal-aware chunk library** produced by NB02.

The searchable representation includes:
- regulation title
- legal location
- article
- heading
- chunk text

This is generic metadata enrichment; it is not specific to any benchmark question.


In [3]:
def safe(v):
    return "" if v is None else str(v)

def chunk_text(c):
    parts = [
        safe(c.get("regulation_title")),
        safe(c.get("location")),
        safe(c.get("article")),
        safe(c.get("heading")),
        safe(c.get("text")),
    ]
    return " | ".join(p for p in parts if p).strip()

legal_texts = [chunk_text(c) for c in legal_chunks]
assert all(legal_texts)

print("Example searchable chunk:")
print(legal_texts[0][:1200])


Example searchable chunk:
COMMISSION REGULATION (EU) 2023/915 | Article 1 — Definitions | Article 1 | Definitions | Article 1 Definitions For the purposes of this Regulation, the following definitions apply: (a) ‘food’ means food as defined in Article 2 of Regulation (EC) No 178/2002 of the European Parliament and of the Council ( 1 ); (b) ‘food business operator’ means food business operator as defined in Article 3, point 3, of Regulation (EC) No 178/2002; (c) ‘placing on the market’ means placing on the market as defined in Article 3, point 8, of Regulation (EC) No 178/2002; (d) ‘final consumer’ means final consumer as defined in Article 3, point 18, of Regulation (EC) No 178/2002; (e) ‘processing’ means processing as defined in Article 2(1), point (m) of Regulation (EC) No 852/2004 of the European Parliament and of the Council ( 2 ); (f) ‘unprocessed products’ means unprocessed products as defined in Article 2(1), point (n), of Regulation (EC) No 852/2004; and (g) ‘processed product

## 2. Q1–Q6 evidence audit

These support groups reflect the **Verified Reference / known legal basis** and are used only to answer:

> Is the necessary evidence already present somewhere in the chunk library?

They do **not** influence retrieval ranking.

If a support group is absent from the library, the likely problem is parsing/chunking/source coverage.

If it exists in the library but not in Top-k, the likely problem is retrieval/ranking.


In [4]:
# Each support group is satisfied if at least one chunk matches one of its alternatives.
# These are diagnostic predicates only — NOT retrieval features.

SUPPORT_GROUPS = {
    "Q1": {
        "contaminant_limits_2023_915": [
            {"file": "EU_2023_915", "terms": ["aflatoxin"]}
        ],
        "special_import_2019_1793": [
            {"file": "EU_2019_1793", "terms": ["dried figs", "türkiye"]}
        ],
        "general_food_law_178_2002": [
            {"file": "EU_2002_178", "terms": ["food safety"]}
        ],
        "official_controls_2017_625": [
            {"file": "EU_2017_625", "terms": ["official controls"]}
        ],
    },

    "Q2": {
        "dried_figs_6_10_row": [
            {"file": "EU_2023_915", "terms": ["1.1.3", "dried figs", "6,0", "10,0"]},
            {"file": "EU_2023_915", "terms": ["1.1.3", "dried figs", "6.0", "10.0"]},
        ],
    },

    "Q3": {
        "dried_figs_scope_row": [
            {"file": "EU_2023_915", "terms": ["1.1.3", "dried figs", "80", "article 3"]}
        ],
        "article_3_rule": [
            {"file": "EU_2023_915", "article": "Article 3"}
        ],
    },

    "Q4": {
        "turkiye_figs_annex_ii": [
            {"file": "EU_2019_1793", "terms": ["türkiye", "dried figs", "aflatoxins", "ochratoxin", "30%"]}
        ],
    },

    "Q5": {
        "annex_ii_scope_and_30pct": [
            {"file": "EU_2019_1793", "terms": ["türkiye", "dried figs", "30%"]}
        ],
        "identification_code": [
            {"file": "EU_2019_1793", "article": "Article 9"}
        ],
        "sampling_analysis_results": [
            {"file": "EU_2019_1793", "article": "Article 10"}
        ],
        "official_certificate": [
            {"file": "EU_2019_1793", "article": "Article 11"}
        ],
        "ched": [
            {"file": "EU_2017_625", "article": "Article 56"}
        ],
        "prior_notification": [
            {"file": "EU_2019__1013", "terms": ["prior notification"]}
        ],
    },

    "Q6": {
        "fbo_article_19": [
            {"file": "EU_2002_178", "article": "Article 19"}
        ],
        "authority_article_66": [
            {"file": "EU_2017_625", "article": "Article 66"}
        ],
    },
}

def matches_rule(c, rule):
    file_ok = True
    article_ok = True
    terms_ok = True

    if rule.get("file"):
        file_ok = rule["file"].casefold() in safe(c.get("file")).casefold()

    if rule.get("article"):
        article_ok = rule["article"].casefold() == safe(c.get("article")).casefold()

    if rule.get("terms"):
        haystack = chunk_text(c).casefold()
        terms_ok = all(t.casefold() in haystack for t in rule["terms"])

    return file_ok and article_ok and terms_ok

def group_matches(c, alternatives):
    return any(matches_rule(c, rule) for rule in alternatives)

audit_rows = []
for qid, groups in SUPPORT_GROUPS.items():
    for group_name, alternatives in groups.items():
        matched = [c for c in legal_chunks if group_matches(c, alternatives)]
        audit_rows.append({
            "qid": qid,
            "support_group": group_name,
            "present_in_library": bool(matched),
            "n_matching_chunks": len(matched),
            "example_chunk_id": matched[0]["chunk_id"] if matched else "",
            "example_location": matched[0].get("location", "") if matched else "",
        })

library_audit = pd.DataFrame(audit_rows)
display(library_audit)

missing = library_audit[~library_audit["present_in_library"]]
if missing.empty:
    print("PASS: All Q1–Q6 diagnostic support groups exist somewhere in the chunk library.")
else:
    print("ATTENTION: Some expected support groups are absent from the library.")
    display(missing)


,qid,support_group,present_in_library,n_matching_chunks,example_chunk_id,example_location
0,Q1,contaminant_limits_2023_915,True,21,legal::EU_2023_915_consolidated_2026-08-19.htm...,"Table 3, row 1"
1,Q1,special_import_2019_1793,True,1,legal::EU_2019_1793_consolidated_2026-06-30.ht...,"Annex II, Section 1, Row 14 — Türkiye dried-fi..."
2,Q1,general_food_law_178_2002,True,9,legal::EU_2002_178_consolidated_2026-01-01.htm...,Article 1 — Aim and scope
3,Q1,official_controls_2017_625,True,137,legal::EU_2017_625_consolidated_2025-01-05.htm...,Article 1 — Subject matter and scope
4,Q2,dried_figs_6_10_row,True,1,legal::EU_2023_915_consolidated_2026-08-19.htm...,"Table 3, row 6"
5,Q3,dried_figs_scope_row,True,1,legal::EU_2023_915_consolidated_2026-08-19.htm...,"Table 3, row 6"
6,Q3,article_3_rule,True,1,legal::EU_2023_915_consolidated_2026-08-19.htm...,"Article 3 — Dried, diluted, processed and comp..."
7,Q4,turkiye_figs_annex_ii,True,1,legal::EU_2019_1793_consolidated_2026-06-30.ht...,"Annex II, Section 1, Row 14 — Türkiye dried-fi..."
8,Q5,annex_ii_scope_and_30pct,True,1,legal::EU_2019_1793_consolidated_2026-06-30.ht...,"Annex II, Section 1, Row 14 — Türkiye dried-fi..."
9,Q5,identification_code,True,1,legal::EU_2019_1793_consolidated_2026-06-30.ht...,Article 9 — Identification code


PASS: All Q1–Q6 diagnostic support groups exist somewhere in the chunk library.


### How to interpret the audit

- **Present in library = True, but not retrieved:** ranking/retrieval problem.
- **Present in library = False:** inspect parsing, row grouping, or missing source coverage first.


## 3. Reproduce the current dense baseline

This deliberately mirrors the existing NB03 approach:

- `paraphrase-multilingual-MiniLM-L12-v2`
- normalized embeddings
- dot product = cosine similarity
- Top-k ranking


In [5]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)

query_ids = [f"Q{i}" for i in range(1, 7)]
query_texts = [questions[qid] for qid in query_ids]

legal_embeddings = model.encode(
    legal_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
query_embeddings = model.encode(
    query_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

dense_scores_by_qid = {}
dense_rank_by_qid = {}

for i, qid in enumerate(query_ids):
    scores = legal_embeddings @ query_embeddings[i]
    order = np.argsort(scores)[::-1]
    dense_scores_by_qid[qid] = scores
    dense_rank_by_qid[qid] = order

print("Dense baseline ready.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Dense baseline ready.


## 4. Add a lexical BM25 retriever

Why add lexical retrieval?

Legal questions often depend on exact strings such as:
- `1.1.3`
- `6,0`
- `10,0`
- `Article 66`
- `CHED`
- `official certificate`

Dense embeddings can rank semantically similar but legally wrong rows highly. BM25 gives exact legal terminology another route into the candidate set.


In [6]:
TOKEN_RE = re.compile(r"[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?")

def tokenize(text):
    return TOKEN_RE.findall(text.casefold())

tokenized_corpus = [tokenize(t) for t in legal_texts]
bm25 = BM25Okapi(tokenized_corpus)

bm25_scores_by_qid = {}
bm25_rank_by_qid = {}

for qid in query_ids:
    scores = np.asarray(bm25.get_scores(tokenize(questions[qid])), dtype=float)
    order = np.argsort(scores)[::-1]
    bm25_scores_by_qid[qid] = scores
    bm25_rank_by_qid[qid] = order

print("BM25 lexical retriever ready.")


BM25 lexical retriever ready.


## 5. Hybrid rank fusion with RRF

We use **Reciprocal Rank Fusion (RRF)** rather than directly adding dense and BM25 scores because their score scales are different.

No benchmark-specific weights or keywords are used in the ranking function.


In [7]:
RRF_K = 60
CANDIDATE_DEPTH = 50

def reciprocal_rank_fusion(qid, candidate_depth=CANDIDATE_DEPTH, rrf_k=RRF_K):
    dense_order = dense_rank_by_qid[qid][:candidate_depth]
    bm25_order = bm25_rank_by_qid[qid][:candidate_depth]

    fused = {}

    for rank, idx in enumerate(dense_order, start=1):
        fused[int(idx)] = fused.get(int(idx), 0.0) + 1.0 / (rrf_k + rank)

    for rank, idx in enumerate(bm25_order, start=1):
        fused[int(idx)] = fused.get(int(idx), 0.0) + 1.0 / (rrf_k + rank)

    return sorted(fused, key=fused.get, reverse=True), fused

hybrid_rank_by_qid = {}
hybrid_scores_by_qid = {}

for qid in query_ids:
    order, scores = reciprocal_rank_fusion(qid)
    hybrid_rank_by_qid[qid] = np.array(order, dtype=int)
    hybrid_scores_by_qid[qid] = scores

print("Hybrid RRF ranking ready.")


Hybrid RRF ranking ready.


## 6. Light legal-source diversification

Q5 and Q6 need evidence from multiple legal provisions. A plain Top-5 can be filled by several near-duplicate chunks from the same article.

The rule below is intentionally generic:

- at most **2 chunks from the same file + article/location group**
- no QID-specific source forcing
- no manually injected gold chunk IDs


In [8]:
def source_group(c):
    file_name = safe(c.get("file"))
    article = safe(c.get("article"))
    location = safe(c.get("location"))

    # Prefer Article identity; otherwise use location.
    legal_unit = article if article else location
    return (file_name, legal_unit)

def diversified_topk(order, k=5, max_per_source_group=2):
    selected = []
    counts = {}

    for idx in order:
        c = legal_chunks[int(idx)]
        group = source_group(c)
        n = counts.get(group, 0)

        if n >= max_per_source_group:
            continue

        selected.append(int(idx))
        counts[group] = n + 1

        if len(selected) >= k:
            break

    return selected

hybrid_diverse_rank_by_qid = {
    qid: diversified_topk(hybrid_rank_by_qid[qid], k=10, max_per_source_group=2)
    for qid in query_ids
}

print("Diversified hybrid ranking ready.")


Diversified hybrid ranking ready.


## 7. Evaluate evidence coverage

This is the key diagnostic.

We compare:
- `dense_baseline`
- `hybrid_rrf`
- `hybrid_rrf_diverse`

at **Top-5** and **Top-10**.

A support group is counted only if an actually retrieved chunk satisfies the diagnostic evidence rule.


In [9]:
def top_indices(qid, method, k):
    if method == "dense_baseline":
        return [int(i) for i in dense_rank_by_qid[qid][:k]]

    if method == "hybrid_rrf":
        return [int(i) for i in hybrid_rank_by_qid[qid][:k]]

    if method == "hybrid_rrf_diverse":
        # Re-run selection at the requested k so Top-5 and Top-10 are consistent.
        return diversified_topk(
            hybrid_rank_by_qid[qid],
            k=k,
            max_per_source_group=2,
        )

    raise ValueError(method)

def coverage_for(qid, method, k):
    idxs = top_indices(qid, method, k)
    retrieved = [legal_chunks[i] for i in idxs]

    groups = SUPPORT_GROUPS[qid]
    hit_groups = []
    missed_groups = []

    for group_name, alternatives in groups.items():
        hit = any(group_matches(c, alternatives) for c in retrieved)
        (hit_groups if hit else missed_groups).append(group_name)

    return {
        "qid": qid,
        "method": method,
        "k": k,
        "groups_total": len(groups),
        "groups_hit": len(hit_groups),
        "coverage": len(hit_groups) / len(groups),
        "hit_groups": " | ".join(hit_groups),
        "missed_groups": " | ".join(missed_groups),
    }

coverage_rows = []
for qid in query_ids:
    for method in ["dense_baseline", "hybrid_rrf", "hybrid_rrf_diverse"]:
        for k in [5, 10]:
            coverage_rows.append(coverage_for(qid, method, k))

coverage_df = pd.DataFrame(coverage_rows)

display(
    coverage_df.sort_values(["qid", "k", "method"]).reset_index(drop=True)
)

print("\nMean support-group coverage:")
display(
    coverage_df.groupby(["method", "k"], as_index=False)["coverage"].mean()
        .sort_values(["k", "coverage"], ascending=[True, False])
)


,qid,method,k,groups_total,groups_hit,coverage,hit_groups,missed_groups
0,Q1,dense_baseline,5,4,1,0.250000,special_import_2019_1793,contaminant_limits_2023_915 | general_food_law...
1,Q1,hybrid_rrf,5,4,2,0.500000,contaminant_limits_2023_915 | special_import_2...,general_food_law_178_2002 | official_controls_...
2,Q1,hybrid_rrf_diverse,5,4,2,0.500000,contaminant_limits_2023_915 | special_import_2...,general_food_law_178_2002 | official_controls_...
3,Q1,dense_baseline,10,4,1,0.250000,special_import_2019_1793,contaminant_limits_2023_915 | general_food_law...
4,Q1,hybrid_rrf,10,4,2,0.500000,contaminant_limits_2023_915 | special_import_2...,general_food_law_178_2002 | official_controls_...
5,Q1,hybrid_rrf_diverse,10,4,2,0.500000,contaminant_limits_2023_915 | special_import_2...,general_food_law_178_2002 | official_controls_...
6,Q2,dense_baseline,5,1,0,0.000000,,dried_figs_6_10_row
7,Q2,hybrid_rrf,5,1,0,0.000000,,dried_figs_6_10_row
8,Q2,hybrid_rrf_diverse,5,1,0,0.000000,,dried_figs_6_10_row
9,Q2,dense_baseline,10,1,0,0.000000,,dried_figs_6_10_row



Mean support-group coverage:


,method,k,coverage
2,hybrid_rrf,5,0.527778
4,hybrid_rrf_diverse,5,0.527778
0,dense_baseline,5,0.291667
3,hybrid_rrf,10,0.694444
5,hybrid_rrf_diverse,10,0.694444
1,dense_baseline,10,0.291667


## 8. Inspect Q1–Q6 retrieved chunks side-by-side

This is the human-review step. Automatic support rules are only diagnostics.

Look for:
- correct regulation
- exact article/annex/table row
- conditions/exceptions in the same chunk
- duplicate/near-duplicate evidence
- irrelevant but semantically similar table rows


In [10]:
def build_result_rows(qid, method, k=5):
    idxs = top_indices(qid, method, k)
    rows = []

    for rank, idx in enumerate(idxs, start=1):
        c = legal_chunks[idx]

        if method == "dense_baseline":
            score = float(dense_scores_by_qid[qid][idx])
        elif method.startswith("hybrid"):
            score = float(hybrid_scores_by_qid[qid].get(idx, 0.0))
        else:
            score = np.nan

        rows.append({
            "qid": qid,
            "method": method,
            "rank": rank,
            "score": score,
            "chunk_id": c.get("chunk_id", ""),
            "file": c.get("file", ""),
            "location": c.get("location", ""),
            "article": c.get("article", ""),
            "text_preview": re.sub(r"\s+", " ", safe(c.get("text")))[:700],
        })

    return rows

comparison_rows = []
for qid in query_ids:
    for method in ["dense_baseline", "hybrid_rrf_diverse"]:
        comparison_rows.extend(build_result_rows(qid, method, k=5))

comparison_df = pd.DataFrame(comparison_rows)

for qid in query_ids:
    print("\n" + "="*100)
    print(qid, "—", questions[qid])
    display(
        comparison_df[comparison_df["qid"] == qid][
            ["method", "rank", "score", "file", "location", "article", "text_preview"]
        ].reset_index(drop=True)
    )



Q1 — A shipment of dried figs originating from Türkiye is found to contain aflatoxin B1. Which EU legislation is relevant to this case?


,method,rank,score,file,location,article,text_preview
0,dense_baseline,1,0.646495,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,dense_baseline,2,0.527298,EU_2019_1793_consolidated_2026-06-30.html,"Table 4, row 178",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
2,dense_baseline,3,0.509490,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 3",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
3,dense_baseline,4,0.502502,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 17",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
4,dense_baseline,5,0.502281,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 19",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
5,hybrid_rrf_diverse,1,0.032787,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
6,hybrid_rrf_diverse,2,0.028177,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 6",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
7,hybrid_rrf_diverse,3,0.026667,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 12",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
8,hybrid_rrf_diverse,4,0.025859,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 14",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
9,hybrid_rrf_diverse,5,0.024923,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 10",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...



Q2 — What is the applicable EU maximum level for aflatoxin B1 and total aflatoxins in dried figs intended for direct human consumption?


,method,rank,score,file,location,article,text_preview
0,dense_baseline,1,0.628532,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 16",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
1,dense_baseline,2,0.627697,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 22",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
2,dense_baseline,3,0.627266,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 11",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
3,dense_baseline,4,0.627220,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 17",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
4,dense_baseline,5,0.627183,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 14",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
5,hybrid_rrf_diverse,1,0.031498,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 11",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
6,hybrid_rrf_diverse,2,0.031025,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 13",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
7,hybrid_rrf_diverse,3,0.029877,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 14",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
8,hybrid_rrf_diverse,4,0.029857,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 9",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
9,hybrid_rrf_diverse,5,0.029274,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 7",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...



Q3 — Does the applicable limit change depending on whether the dried figs are intended for direct consumption or further processing/sorting?


...

,method,rank,score,file,location,article,text_preview
0,dense_baseline,1,0.682997,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,dense_baseline,2,0.540433,EU_2017_625_consolidated_2025-01-05.html,Article 73 — Approval of pre-export controls p...,Article 73,may only be granted to a third country if the ...
2,dense_baseline,3,0.531223,EU_2019_1793_consolidated_2026-06-30.html,"Table 5, row 169",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
3,dense_baseline,4,0.524875,EU_2017_625_consolidated_2025-01-05.html,Article 18 — Specific rules on official contro...,Article 18,es referred to in Articles 137(2) and 138(2); ...
4,dense_baseline,5,0.517911,EU_2023_915_consolidated_2026-08-19.html,Article 7 — Derogations from Article 2,Article 7,Article 7 Derogations from Article 2 1. By way...
5,hybrid_rrf_diverse,1,0.032787,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
6,hybrid_rrf_diverse,2,0.030777,EU_2017_625_consolidated_2025-01-05.html,Article 18 — Specific rules on official contro...,Article 18,an official veterinarian in regard to the ante...
7,hybrid_rrf_diverse,3,0.027418,EU_2017_625_consolidated_2025-01-05.html,Article 18 — Specific rules on official contro...,Article 18,"consumption, the health mark shall be applied ..."
8,hybrid_rrf_diverse,4,0.023669,EU_2017_625_consolidated_2025-01-05.html,Article 46 — Samples taken on animals and good...,Article 46,Article 46 Samples taken on animals and goods ...
9,hybrid_rrf_diverse,5,0.023088,EU_2017_625_consolidated_2025-01-05.html,"Article 74 — Non-compliance with, and withdraw...",Article 74,"Article 74 Non-compliance with, and withdrawal..."



Q5 — If so, what controls, sampling/testing requirements, or documentation requirements currently apply?


,method,rank,score,file,location,article,text_preview
0,dense_baseline,1,0.458844,EU_2017_625_consolidated_2025-01-05.html,"Article 34 — Methods used for sampling, analys...",Article 34,"xists, the relevant national reference laborat..."
1,dense_baseline,2,0.437452,EU_2017_625_consolidated_2025-01-05.html,Article 40 — Derogations from the condition fo...,Article 40,ormance in the inter-laboratory comparative te...
2,dense_baseline,3,0.411042,EU_2017_625_consolidated_2025-01-05.html,"Article 34 — Methods used for sampling, analys...",Article 34,"Article 34 Methods used for sampling, analyses..."
3,dense_baseline,4,0.403383,EU_2017_625_consolidated_2025-01-05.html,"Article 54 — Frequency of documentary checks, ...",Article 54,tific assessments; and (vi) any other informat...
4,dense_baseline,5,0.390842,EU_2017_625_consolidated_2025-01-05.html,Article 86 — General requirements concerning o...,Article 86,Article 86 General requirements concerning off...
5,hybrid_rrf_diverse,1,0.032266,EU_2017_625_consolidated_2025-01-05.html,"Article 34 — Methods used for sampling, analys...",Article 34,"Article 34 Methods used for sampling, analyses..."
6,hybrid_rrf_diverse,2,0.030886,EU_2017_625_consolidated_2025-01-05.html,"Article 34 — Methods used for sampling, analys...",Article 34,"xists, the relevant national reference laborat..."
7,hybrid_rrf_diverse,3,0.027444,EU_2019_1793_consolidated_2026-06-30.html,Article 3 — Sampling and analyses,Article 3,Article 3 Sampling and analyses The sampling a...
8,hybrid_rrf_diverse,4,0.026882,EU_2019_1793_consolidated_2026-06-30.html,Article 10 — Results of sampling and analyses ...,Article 10,Article 10 Results of sampling and analyses pe...
9,hybrid_rrf_diverse,5,0.026161,EU_2017_625_consolidated_2025-01-05.html,Article 100 — Designation of national referenc...,Article 100,"have contractual access to, suitably qualified..."



Q6 — Which authorities or food business operators are responsible for the relevant actions when a non-compliant consignment is detected?


,method,rank,score,file,location,article,text_preview
0,dense_baseline,1,0.731057,EU_2002_178_consolidated_2026-01-01.html,Article 19 — Responsibilities for food: food b...,Article 19,ities of the action taken to prevent risks to ...
1,dense_baseline,2,0.700700,EU_2002_178_consolidated_2026-01-01.html,Article 19 — Responsibilities for food: food b...,Article 19,Article 19 Responsibilities for food: food bus...
2,dense_baseline,3,0.663225,EU_2002_178_consolidated_2026-01-01.html,Article 17 — Responsibilities,Article 17,Article 17 Responsibilities 1. Food and feed b...
3,dense_baseline,4,0.661497,EU_2002_178_consolidated_2026-01-01.html,Article 20 — Responsibilities for feed: feed b...,Article 20,atisfy the feed safety requirements. It shall ...
4,dense_baseline,5,0.638976,EU_2002_178_consolidated_2026-01-01.html,Article 20 — Responsibilities for feed: feed b...,Article 20,Article 20 Responsibilities for feed: feed bus...
5,hybrid_rrf_diverse,1,0.032258,EU_2002_178_consolidated_2026-01-01.html,Article 19 — Responsibilities for food: food b...,Article 19,Article 19 Responsibilities for food: food bus...
6,hybrid_rrf_diverse,2,0.031778,EU_2002_178_consolidated_2026-01-01.html,Article 20 — Responsibilities for feed: feed b...,Article 20,Article 20 Responsibilities for feed: feed bus...
7,hybrid_rrf_diverse,3,0.030478,EU_2002_178_consolidated_2026-01-01.html,Article 19 — Responsibilities for food: food b...,Article 19,ities of the action taken to prevent risks to ...
8,hybrid_rrf_diverse,4,0.029514,EU_2002_178_consolidated_2026-01-01.html,Article 20 — Responsibilities for feed: feed b...,Article 20,atisfy the feed safety requirements. It shall ...
9,hybrid_rrf_diverse,5,0.028139,EU_2017_625_consolidated_2025-01-05.html,Article 148 — Relation with Regulations (EC) N...,Article 148,Article 148 Relation with Regulations (EC) No ...


## 9. Failure classification

This converts the observations into the distinction we need before changing NB02/NB03.

- **source_or_chunk_failure**: expected support evidence is absent from the library.
- **ranking_failure**: evidence exists in the library but current dense Top-5 misses it.
- **topk_capacity_or_diversity_issue**: evidence appears only when using more candidates/diversification.
- **retrieval_ok**: baseline Top-5 already covers the diagnostic evidence.


In [11]:
failure_rows = []

for qid in query_ids:
    groups = SUPPORT_GROUPS[qid]

    for group_name, alternatives in groups.items():
        present_in_library = any(
            group_matches(c, alternatives)
            for c in legal_chunks
        )

        dense5 = any(
            group_matches(legal_chunks[i], alternatives)
            for i in top_indices(qid, "dense_baseline", 5)
        )

        hybrid5 = any(
            group_matches(legal_chunks[i], alternatives)
            for i in top_indices(qid, "hybrid_rrf_diverse", 5)
        )

        hybrid10 = any(
            group_matches(legal_chunks[i], alternatives)
            for i in top_indices(qid, "hybrid_rrf_diverse", 10)
        )

        if not present_in_library:
            diagnosis = "source_or_chunk_failure"
        elif dense5:
            diagnosis = "retrieval_ok"
        elif hybrid5:
            diagnosis = "ranking_failure_improved_by_hybrid"
        elif hybrid10:
            diagnosis = "topk_capacity_or_diversity_issue"
        else:
            diagnosis = "ranking_failure_not_fixed"

        failure_rows.append({
            "qid": qid,
            "support_group": group_name,
            "present_in_library": present_in_library,
            "dense_top5_hit": dense5,
            "hybrid_diverse_top5_hit": hybrid5,
            "hybrid_diverse_top10_hit": hybrid10,
            "diagnosis": diagnosis,
        })

failure_df = pd.DataFrame(failure_rows)
display(failure_df)

print("\nDiagnosis counts:")
display(failure_df["diagnosis"].value_counts().rename_axis("diagnosis").reset_index(name="n"))


,qid,support_group,present_in_library,dense_top5_hit,hybrid_diverse_top5_hit,hybrid_diverse_top10_hit,diagnosis
0,Q1,contaminant_limits_2023_915,True,False,True,True,ranking_failure_improved_by_hybrid
1,Q1,special_import_2019_1793,True,True,True,True,retrieval_ok
2,Q1,general_food_law_178_2002,True,False,False,False,ranking_failure_not_fixed
3,Q1,official_controls_2017_625,True,False,False,False,ranking_failure_not_fixed
4,Q2,dried_figs_6_10_row,True,False,False,True,topk_capacity_or_diversity_issue
5,Q3,dried_figs_scope_row,True,False,True,True,ranking_failure_improved_by_hybrid
6,Q3,article_3_rule,True,False,True,True,ranking_failure_improved_by_hybrid
7,Q4,turkiye_figs_annex_ii,True,True,True,True,retrieval_ok
8,Q5,annex_ii_scope_and_30pct,True,False,False,False,ranking_failure_not_fixed
9,Q5,identification_code,True,False,False,False,ranking_failure_not_fixed



Diagnosis counts:


,diagnosis,n
0,ranking_failure_not_fixed,8
1,ranking_failure_improved_by_hybrid,4
2,retrieval_ok,3
3,topk_capacity_or_diversity_issue,1


## 10. Save candidate v2 retrieval results

This file is **not yet the official NB04 input**.

First review Q1–Q6 manually. If the hybrid method is clearly better and does not introduce unacceptable noise, then apply the same frozen algorithm to all Q1–Q10 and create a new official retrieval version.


In [12]:
candidate_rows = []

for qid in questions:
    # Build embeddings/ranks for Q7–Q10 lazily if needed.
    if qid not in dense_rank_by_qid:
        q_emb = model.encode(
            [questions[qid]],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )[0]

        dense_scores = legal_embeddings @ q_emb
        dense_rank_by_qid[qid] = np.argsort(dense_scores)[::-1]
        dense_scores_by_qid[qid] = dense_scores

        bm_scores = np.asarray(bm25.get_scores(tokenize(questions[qid])), dtype=float)
        bm25_rank_by_qid[qid] = np.argsort(bm_scores)[::-1]
        bm25_scores_by_qid[qid] = bm_scores

        order, fused = reciprocal_rank_fusion(qid)
        hybrid_rank_by_qid[qid] = np.array(order, dtype=int)
        hybrid_scores_by_qid[qid] = fused

    idxs = diversified_topk(
        hybrid_rank_by_qid[qid],
        k=5,
        max_per_source_group=2,
    )

    for rank, idx in enumerate(idxs, start=1):
        c = legal_chunks[idx]

        candidate_rows.append({
            "qid": qid,
            "question": questions[qid],
            "benchmark_id": benchmark["benchmark_id"],
            "benchmark_sha256": BENCHMARK_SHA256,
            "strategy": "legal_aware_hybrid_rrf_diverse_v2",
            "rank": rank,
            "score": float(hybrid_scores_by_qid[qid].get(idx, 0.0)),
            "chunk_id": c.get("chunk_id", ""),
            "file": c.get("file", ""),
            "version_date": c.get("version_date", ""),
            "location": c.get("location", ""),
            "article": c.get("article", ""),
            "heading": c.get("heading", ""),
            "text_preview": re.sub(r"\s+", " ", safe(c.get("text")))[:650],
        })

candidate_df = pd.DataFrame(candidate_rows)

CANDIDATE_PATH = DATA_DIR / "retrieval_results_nb03_hybrid_v2_candidate.csv"
AUDIT_PATH = DATA_DIR / "retrieval_library_audit_q1_q6.csv"
COVERAGE_PATH = DATA_DIR / "retrieval_v2_coverage_q1_q6.csv"
FAILURE_PATH = DATA_DIR / "retrieval_v2_failure_diagnosis_q1_q6.csv"

candidate_df.to_csv(CANDIDATE_PATH, index=False, encoding="utf-8-sig")
library_audit.to_csv(AUDIT_PATH, index=False, encoding="utf-8-sig")
coverage_df.to_csv(COVERAGE_PATH, index=False, encoding="utf-8-sig")
failure_df.to_csv(FAILURE_PATH, index=False, encoding="utf-8-sig")

print("Saved:")
print(" -", CANDIDATE_PATH)
print(" -", AUDIT_PATH)
print(" -", COVERAGE_PATH)
print(" -", FAILURE_PATH)


Saved:
 - data\retrieval_results_nb03_hybrid_v2_candidate.csv
 - data\retrieval_library_audit_q1_q6.csv
 - data\retrieval_v2_coverage_q1_q6.csv
 - data\retrieval_v2_failure_diagnosis_q1_q6.csv


## 11. Decision checklist

Before using candidate v2 in NB04:

- [ ] Q2: Does Top-5 retrieve the `1.1.3 Dried figs | 6,0 | 10,0` row?
- [ ] Q3: Does retrieval include the dried-fig 80% / Article 3 condition rather than generic dried-fruit sorting rules?
- [ ] Q4: Is the previously good Türkiye dried-fig Annex II row still retained?
- [ ] Q5: Does Top-5/Top-10 cover multiple operational requirements rather than generic laboratory-control articles only?
- [ ] Q6: Are both Article 19 (FBO) and Article 66 (competent authority/border action) represented?
- [ ] Did the algorithm improve results through general hybrid retrieval rather than QID-specific boosts?
- [ ] Are Q7–Q10 inspected before freezing the candidate as the official Build02 v2 retrieval?

### Recommended stop rule

Do **one** targeted retrieval-improvement round.  
Do not keep optimizing until all 10 development questions are perfect.

After review:
1. Freeze the retrieval method.
2. Run the new GPT-6 Luna NB04 from Q1–Q10.
3. Treat the current dried-fig benchmark as a development benchmark.
4. Later test generalization on a different unseen food-regulation case.


# NB03c — Final Build02 experiment: query decomposition / multi-query retrieval

This section extends NB03b **without changing the legal corpus or the benchmark questions**.

## Why this experiment?

NB03b showed that:

- the needed evidence for Q1–Q6 is generally present in the chunk library;
- Q3 improved with hybrid retrieval;
- Q5 and Q6 still suffer because one complex question may require evidence from several legal provisions.

The final Build02 experiment asks:

> Can a complex regulatory question be split into several smaller search intents, retrieve evidence for each intent, then merge the results?

### Anti-overfitting rule

The decomposition below may use:
- the **question wording itself**, and
- the **shared case context** already stated by the benchmark: Türkiye-origin dried figs / aflatoxin B1.

It does **not** contain:
- regulation numbers,
- article numbers,
- expected numeric answers,
- manually selected gold chunk IDs.

Therefore the retriever is not directly told the Verified Reference answer.


## 12. Create multi-query decompositions

Every question keeps the original query.

For compound questions, we additionally create narrower subqueries corresponding to the explicit facets in the wording.

Examples:

- Q5 → controls / sampling-testing / documentation
- Q6 → food-business-operator responsibility / competent-authority responsibility

The same scenario context is attached to each subquery so that phrases such as **"If so"** do not lose the subject of the case.


In [13]:
SCENARIO_CONTEXT = (
    "EU regulatory assessment for a shipment of dried figs originating from Türkiye "
    "involving aflatoxin B1."
)

def decompose_question(qid, question):
    """
    Deterministic, answer-agnostic decomposition.
    Uses only the wording of the question + shared benchmark scenario context.
    """
    q = " ".join(question.split())
    q_lower = q.casefold()

    subqueries = [f"{SCENARIO_CONTEXT} {q}"]

    # Compound requirement facets.
    if "controls, sampling/testing requirements, or documentation requirements" in q_lower:
        subqueries.extend([
            f"{SCENARIO_CONTEXT} What official controls currently apply to this consignment?",
            f"{SCENARIO_CONTEXT} What sampling and testing requirements currently apply to this consignment?",
            f"{SCENARIO_CONTEXT} What documentation requirements currently apply to this consignment?",
        ])

    # Responsibility facets.
    if "authorities or food business operators" in q_lower:
        subqueries.extend([
            f"{SCENARIO_CONTEXT} What responsibilities do food business operators have when the consignment is non-compliant?",
            f"{SCENARIO_CONTEXT} What actions or responsibilities do competent authorities have when the consignment is non-compliant?",
        ])

    # Direct-consumption vs processing/sorting comparison.
    if "direct consumption or further processing/sorting" in q_lower:
        subqueries.extend([
            f"{SCENARIO_CONTEXT} What rules apply to dried figs intended for direct human consumption?",
            f"{SCENARIO_CONTEXT} What rules apply to dried figs intended for further processing or sorting?",
        ])

    # Applicable legislation question.
    if "which eu legislation is relevant" in q_lower:
        subqueries.extend([
            f"{SCENARIO_CONTEXT} Which EU legislation governs aflatoxin maximum levels?",
            f"{SCENARIO_CONTEXT} Which EU legislation governs official controls or import conditions?",
            f"{SCENARIO_CONTEXT} Which EU general food-law provisions are relevant to non-compliant food?",
        ])

    # Numeric maximum-level question.
    if "maximum level for aflatoxin b1 and total aflatoxins" in q_lower:
        subqueries.extend([
            f"{SCENARIO_CONTEXT} Maximum level aflatoxin B1 in dried figs intended for direct human consumption.",
            f"{SCENARIO_CONTEXT} Maximum level total aflatoxins B1 B2 G1 G2 in dried figs intended for direct human consumption.",
        ])

    # Increased-controls / special-import-condition question.
    if "increased official controls or special import conditions" in q_lower:
        subqueries.extend([
            f"{SCENARIO_CONTEXT} Increased official controls for dried figs from Türkiye.",
            f"{SCENARIO_CONTEXT} Special import conditions for dried figs from Türkiye.",
        ])

    # Deduplicate while preserving order.
    seen = set()
    clean = []
    for sq in subqueries:
        key = sq.casefold()
        if key not in seen:
            seen.add(key)
            clean.append(sq)

    return clean


decomposition_rows = []
for qid in [f"Q{i}" for i in range(1, 7)]:
    for n, sq in enumerate(decompose_question(qid, questions[qid]), start=1):
        decomposition_rows.append({
            "qid": qid,
            "subquery_id": f"{qid}_S{n}",
            "subquery": sq,
        })

decomposition_df = pd.DataFrame(decomposition_rows)
display(decomposition_df)


,qid,subquery_id,subquery
0,Q1,Q1_S1,EU regulatory assessment for a shipment of dri...
1,Q1,Q1_S2,EU regulatory assessment for a shipment of dri...
2,Q1,Q1_S3,EU regulatory assessment for a shipment of dri...
3,Q1,Q1_S4,EU regulatory assessment for a shipment of dri...
4,Q2,Q2_S1,EU regulatory assessment for a shipment of dri...
5,Q2,Q2_S2,EU regulatory assessment for a shipment of dri...
6,Q2,Q2_S3,EU regulatory assessment for a shipment of dri...
7,Q3,Q3_S1,EU regulatory assessment for a shipment of dri...
8,Q3,Q3_S2,EU regulatory assessment for a shipment of dri...
9,Q3,Q3_S3,EU regulatory assessment for a shipment of dri...


## 13. Retrieve candidates for each subquery

For every subquery we use the **same generic hybrid retriever from NB03b**:

1. dense semantic retrieval,
2. BM25 lexical retrieval,
3. Reciprocal Rank Fusion (RRF).

Then we merge the rankings from all subqueries using another RRF step.

This creates a final **multi-query ranking**.


In [14]:
MULTIQUERY_PER_QUERY_DEPTH = 30
MULTIQUERY_RRF_K = 60

def dense_rank_for_text(query_text):
    q_emb = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    scores = legal_embeddings @ q_emb
    order = np.argsort(scores)[::-1]
    return order, scores

def bm25_rank_for_text(query_text):
    scores = np.asarray(
        bm25.get_scores(tokenize(query_text)),
        dtype=float,
    )
    order = np.argsort(scores)[::-1]
    return order, scores

def hybrid_rank_for_text(query_text, candidate_depth=50, rrf_k=60):
    dense_order, dense_scores = dense_rank_for_text(query_text)
    bm25_order, bm25_scores = bm25_rank_for_text(query_text)

    fused = {}

    for rank, idx in enumerate(dense_order[:candidate_depth], start=1):
        fused[int(idx)] = fused.get(int(idx), 0.0) + 1.0 / (rrf_k + rank)

    for rank, idx in enumerate(bm25_order[:candidate_depth], start=1):
        fused[int(idx)] = fused.get(int(idx), 0.0) + 1.0 / (rrf_k + rank)

    order = sorted(fused, key=fused.get, reverse=True)
    return order, fused


multiquery_rank_by_qid = {}
multiquery_scores_by_qid = {}
multiquery_subquery_rankings = {}

for qid in [f"Q{i}" for i in range(1, 7)]:
    subqueries = decompose_question(qid, questions[qid])

    aggregate = {}
    sub_rankings = []

    for sq in subqueries:
        order, fused = hybrid_rank_for_text(sq)

        sub_rankings.append({
            "query": sq,
            "order": order,
            "scores": fused,
        })

        # Second-stage RRF across subqueries.
        for rank, idx in enumerate(
            order[:MULTIQUERY_PER_QUERY_DEPTH],
            start=1,
        ):
            aggregate[int(idx)] = (
                aggregate.get(int(idx), 0.0)
                + 1.0 / (MULTIQUERY_RRF_K + rank)
            )

    final_order = sorted(
        aggregate,
        key=aggregate.get,
        reverse=True,
    )

    multiquery_rank_by_qid[qid] = np.array(final_order, dtype=int)
    multiquery_scores_by_qid[qid] = aggregate
    multiquery_subquery_rankings[qid] = sub_rankings

print("Multi-query retrieval ready for Q1–Q6.")


Multi-query retrieval ready for Q1–Q6.


## 14. Apply the same legal-unit diversification

We reuse the same generic diversification rule:

- no more than 2 chunks from the same file + article/location group;
- no forced regulation or article based on QID.


In [15]:
def multiquery_topk(qid, k=5, max_per_source_group=2):
    return diversified_topk(
        multiquery_rank_by_qid[qid],
        k=k,
        max_per_source_group=max_per_source_group,
    )

for qid in [f"Q{i}" for i in range(1, 7)]:
    print(qid, "Top-5 IDs:")
    for i in multiquery_topk(qid, 5):
        print("  ", legal_chunks[i]["chunk_id"])


Q1 Top-5 IDs:
   legal::EU_2019_1793_consolidated_2026-06-30.html::annexII_row14_turkiye_figs
   legal::EU_2019_1793_consolidated_2026-06-30.html::table5_row169::01
   legal::EU_2023_915_consolidated_2026-08-19.html::table3_row6::01
   legal::EU_2023_915_consolidated_2026-08-19.html::table4_row3::01
   legal::EU_2023_915_consolidated_2026-08-19.html::table4_row18::01
Q2 Top-5 IDs:
   legal::EU_2019_1793_consolidated_2026-06-30.html::annexII_row14_turkiye_figs
   legal::EU_2023_915_consolidated_2026-08-19.html::table4_row18::01
   legal::EU_2023_915_consolidated_2026-08-19.html::table4_row3::01
   legal::EU_2023_915_consolidated_2026-08-19.html::table3_row6::01
   legal::EU_2023_915_consolidated_2026-08-19.html::table3_row11::01
Q3 Top-5 IDs:
   legal::EU_2019_1793_consolidated_2026-06-30.html::annexII_row14_turkiye_figs
   legal::EU_2023_915_consolidated_2026-08-19.html::table4_row18::01
   legal::EU_2023_915_consolidated_2026-08-19.html::table4_row3::01
   legal::EU_2019_1793_consolid

## 15. Compare support coverage

We compare:

- current dense baseline,
- single-query hybrid,
- multi-query hybrid.

Both **Top-5 and Top-10** are shown.

Top-10 is diagnostic only. It helps distinguish:
- ranking failure,
- insufficient Top-k capacity,
- true source/chunk problems.


In [16]:
def multiquery_coverage_for(qid, k):
    idxs = multiquery_topk(qid, k=k, max_per_source_group=2)
    retrieved = [legal_chunks[i] for i in idxs]

    groups = SUPPORT_GROUPS[qid]
    hit_groups = []
    missed_groups = []

    for group_name, alternatives in groups.items():
        hit = any(
            group_matches(c, alternatives)
            for c in retrieved
        )
        (hit_groups if hit else missed_groups).append(group_name)

    return {
        "qid": qid,
        "method": "multiquery_hybrid_rrf_diverse",
        "k": k,
        "groups_total": len(groups),
        "groups_hit": len(hit_groups),
        "coverage": len(hit_groups) / len(groups),
        "hit_groups": " | ".join(hit_groups),
        "missed_groups": " | ".join(missed_groups),
    }


multi_cov_rows = []
for qid in [f"Q{i}" for i in range(1, 7)]:
    for k in [5, 10]:
        multi_cov_rows.append(
            multiquery_coverage_for(qid, k)
        )

multi_cov_df = pd.DataFrame(multi_cov_rows)

baseline_compare = coverage_df[
    coverage_df["method"].isin([
        "dense_baseline",
        "hybrid_rrf_diverse",
    ])
].copy()

all_coverage_compare = pd.concat(
    [baseline_compare, multi_cov_df],
    ignore_index=True,
)

display(
    all_coverage_compare
    .sort_values(["qid", "k", "method"])
    .reset_index(drop=True)
)

print("\nMean support-group coverage:")
display(
    all_coverage_compare
    .groupby(["method", "k"], as_index=False)["coverage"]
    .mean()
    .sort_values(["k", "coverage"], ascending=[True, False])
)


,qid,method,k,groups_total,groups_hit,coverage,hit_groups,missed_groups
0,Q1,dense_baseline,5,4,1,0.250000,special_import_2019_1793,contaminant_limits_2023_915 | general_food_law...
1,Q1,hybrid_rrf_diverse,5,4,2,0.500000,contaminant_limits_2023_915 | special_import_2...,general_food_law_178_2002 | official_controls_...
2,Q1,multiquery_hybrid_rrf_diverse,5,4,2,0.500000,contaminant_limits_2023_915 | special_import_2...,general_food_law_178_2002 | official_controls_...
3,Q1,dense_baseline,10,4,1,0.250000,special_import_2019_1793,contaminant_limits_2023_915 | general_food_law...
4,Q1,hybrid_rrf_diverse,10,4,2,0.500000,contaminant_limits_2023_915 | special_import_2...,general_food_law_178_2002 | official_controls_...
5,Q1,multiquery_hybrid_rrf_diverse,10,4,2,0.500000,contaminant_limits_2023_915 | special_import_2...,general_food_law_178_2002 | official_controls_...
6,Q2,dense_baseline,5,1,0,0.000000,,dried_figs_6_10_row
7,Q2,hybrid_rrf_diverse,5,1,0,0.000000,,dried_figs_6_10_row
8,Q2,multiquery_hybrid_rrf_diverse,5,1,1,1.000000,dried_figs_6_10_row,
9,Q2,dense_baseline,10,1,0,0.000000,,dried_figs_6_10_row



Mean support-group coverage:


,method,k,coverage
4,multiquery_hybrid_rrf_diverse,5,0.611111
2,hybrid_rrf_diverse,5,0.527778
0,dense_baseline,5,0.291667
3,hybrid_rrf_diverse,10,0.694444
5,multiquery_hybrid_rrf_diverse,10,0.694444
1,dense_baseline,10,0.291667


## 16. Inspect the actual Top-5 evidence

Automatic coverage scores are not enough.

For each question, inspect whether the final Top-5 is:
- legally relevant,
- non-duplicative,
- sufficiently diverse,
- actually useful to an LLM answer.


In [17]:
multi_result_rows = []

for qid in [f"Q{i}" for i in range(1, 7)]:
    idxs = multiquery_topk(qid, k=5)

    for rank, idx in enumerate(idxs, start=1):
        c = legal_chunks[idx]

        multi_result_rows.append({
            "qid": qid,
            "rank": rank,
            "score": float(
                multiquery_scores_by_qid[qid].get(idx, 0.0)
            ),
            "chunk_id": c.get("chunk_id", ""),
            "file": c.get("file", ""),
            "location": c.get("location", ""),
            "article": c.get("article", ""),
            "text_preview": re.sub(
                r"\s+",
                " ",
                safe(c.get("text")),
            )[:800],
        })

multi_results_df = pd.DataFrame(multi_result_rows)

for qid in [f"Q{i}" for i in range(1, 7)]:
    print("\n" + "=" * 100)
    print(qid, "—", questions[qid])
    display(
        multi_results_df[
            multi_results_df["qid"] == qid
        ][
            [
                "rank",
                "score",
                "file",
                "location",
                "article",
                "text_preview",
            ]
        ].reset_index(drop=True)
    )



Q1 — A shipment of dried figs originating from Türkiye is found to contain aflatoxin B1. Which EU legislation is relevant to this case?


,rank,score,file,location,article,text_preview
0,1,0.063886,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,2,0.060529,EU_2019_1793_consolidated_2026-06-30.html,"Table 5, row 169",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
2,3,0.059276,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 6",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
3,4,0.046343,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 3",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
4,5,0.045139,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 18",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...



Q2 — What is the applicable EU maximum level for aflatoxin B1 and total aflatoxins in dried figs intended for direct human consumption?


,rank,score,file,location,article,text_preview
0,1,0.049180,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,2,0.048387,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 18",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
2,3,0.045831,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 3",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
3,4,0.045295,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 6",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
4,5,0.045147,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 11",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...



Q3 — Does the applicable limit change depending on whether the dried figs are intended for direct consumption or further processing/sorting?


...

,rank,score,file,location,article,text_preview
0,1,0.049180,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,2,0.048387,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 18",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
2,3,0.047619,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 3",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
3,4,0.046635,EU_2019_1793_consolidated_2026-06-30.html,"Table 5, row 169",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
4,5,0.045715,EU_2017_625_consolidated_2025-01-05.html,Article 18 — Specific rules on official contro...,Article 18,an official veterinarian in regard to the ante...



Q5 — If so, what controls, sampling/testing requirements, or documentation requirements currently apply?


,rank,score,file,location,article,text_preview
0,1,0.065574,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,2,0.064260,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 3",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
2,3,0.062531,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 18",NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
3,4,0.062077,EU_2019_1793_consolidated_2026-06-30.html,"Table 5, row 169",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
4,5,0.061087,EU_2019_1793_consolidated_2026-06-30.html,"Table 4, row 178",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...



Q6 — Which authorities or food business operators are responsible for the relevant actions when a non-compliant consignment is detected?


,rank,score,file,location,article,text_preview
0,1,0.048916,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,2,0.044854,EU_2019_1793_consolidated_2026-06-30.html,"Table 5, row 169",NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
2,3,0.044171,EU_2002_178_consolidated_2026-01-01.html,Article 19 — Responsibilities for food: food b...,Article 19,Article 19 Responsibilities for food: food bus...
3,4,0.043750,EU_2002_178_consolidated_2026-01-01.html,Article 20 — Responsibilities for feed: feed b...,Article 20,Article 20 Responsibilities for feed: feed bus...
4,5,0.042899,EU_2017_625_consolidated_2025-01-05.html,Article 91 — Official attestations,Article 91,Article 91 Official attestations 1. When this ...


## 17. Inspect which subquery found which evidence

This helps answer an important Build02 question:

> Did decomposition actually recover a missing legal provision, or did it merely reshuffle the same evidence?


In [18]:
subquery_debug_rows = []

for qid in [f"Q{i}" for i in range(1, 7)]:
    for s_idx, item in enumerate(
        multiquery_subquery_rankings[qid],
        start=1,
    ):
        for rank, idx in enumerate(
            item["order"][:5],
            start=1,
        ):
            c = legal_chunks[int(idx)]

            subquery_debug_rows.append({
                "qid": qid,
                "subquery_id": f"{qid}_S{s_idx}",
                "subquery": item["query"],
                "rank": rank,
                "chunk_id": c.get("chunk_id", ""),
                "file": c.get("file", ""),
                "location": c.get("location", ""),
                "article": c.get("article", ""),
            })

subquery_debug_df = pd.DataFrame(subquery_debug_rows)

for qid in ["Q5", "Q6"]:
    print("\n" + "=" * 100)
    print(qid, "subquery retrieval")
    display(
        subquery_debug_df[
            subquery_debug_df["qid"] == qid
        ].reset_index(drop=True)
    )



Q5 subquery retrieval


,qid,subquery_id,subquery,rank,chunk_id,file,location,article
0,Q5,Q5_S1,EU regulatory assessment for a shipment of dri...,1,legal::EU_2019_1793_consolidated_2026-06-30.ht...,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN
1,Q5,Q5_S1,EU regulatory assessment for a shipment of dri...,2,legal::EU_2023_915_consolidated_2026-08-19.htm...,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 18",NaN
2,Q5,Q5_S1,EU regulatory assessment for a shipment of dri...,3,legal::EU_2023_915_consolidated_2026-08-19.htm...,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 3",NaN
3,Q5,Q5_S1,EU regulatory assessment for a shipment of dri...,4,legal::EU_2019_1793_consolidated_2026-06-30.ht...,EU_2019_1793_consolidated_2026-06-30.html,"Table 5, row 169",NaN
4,Q5,Q5_S1,EU regulatory assessment for a shipment of dri...,5,legal::EU_2017_625_consolidated_2025-01-05.htm...,EU_2017_625_consolidated_2025-01-05.html,"Article 34 — Methods used for sampling, analys...",Article 34
5,Q5,Q5_S2,EU regulatory assessment for a shipment of dri...,1,legal::EU_2019_1793_consolidated_2026-06-30.ht...,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN
6,Q5,Q5_S2,EU regulatory assessment for a shipment of dri...,2,legal::EU_2023_915_consolidated_2026-08-19.htm...,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 3",NaN
7,Q5,Q5_S2,EU regulatory assessment for a shipment of dri...,3,legal::EU_2023_915_consolidated_2026-08-19.htm...,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 6",NaN
8,Q5,Q5_S2,EU regulatory assessment for a shipment of dri...,4,legal::EU_2019_1793_consolidated_2026-06-30.ht...,EU_2019_1793_consolidated_2026-06-30.html,"Table 4, row 178",NaN
9,Q5,Q5_S2,EU regulatory assessment for a shipment of dri...,5,legal::EU_2017_625_consolidated_2025-01-05.htm...,EU_2017_625_consolidated_2025-01-05.html,Article 91 — Official attestations,Article 91



Q6 subquery retrieval


,qid,subquery_id,subquery,rank,chunk_id,file,location,article
0,Q6,Q6_S1,EU regulatory assessment for a shipment of dri...,1,legal::EU_2019_1793_consolidated_2026-06-30.ht...,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN
1,Q6,Q6_S1,EU regulatory assessment for a shipment of dri...,2,legal::EU_2017_625_consolidated_2025-01-05.htm...,EU_2017_625_consolidated_2025-01-05.html,Article 148 — Relation with Regulations (EC) N...,Article 148
2,Q6,Q6_S1,EU regulatory assessment for a shipment of dri...,3,legal::EU_2002_178_consolidated_2026-01-01.htm...,EU_2002_178_consolidated_2026-01-01.html,Article 19 — Responsibilities for food: food b...,Article 19
3,Q6,Q6_S1,EU regulatory assessment for a shipment of dri...,4,legal::EU_2002_178_consolidated_2026-01-01.htm...,EU_2002_178_consolidated_2026-01-01.html,Article 20 — Responsibilities for feed: feed b...,Article 20
4,Q6,Q6_S1,EU regulatory assessment for a shipment of dri...,5,legal::EU_2019_1793_consolidated_2026-06-30.ht...,EU_2019_1793_consolidated_2026-06-30.html,"Table 5, row 169",NaN
5,Q6,Q6_S2,EU regulatory assessment for a shipment of dri...,1,legal::EU_2002_178_consolidated_2026-01-01.htm...,EU_2002_178_consolidated_2026-01-01.html,Article 19 — Responsibilities for food: food b...,Article 19
6,Q6,Q6_S2,EU regulatory assessment for a shipment of dri...,2,legal::EU_2019_1793_consolidated_2026-06-30.ht...,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN
7,Q6,Q6_S2,EU regulatory assessment for a shipment of dri...,3,legal::EU_2002_178_consolidated_2026-01-01.htm...,EU_2002_178_consolidated_2026-01-01.html,Article 19 — Responsibilities for food: food b...,Article 19
8,Q6,Q6_S2,EU regulatory assessment for a shipment of dri...,4,legal::EU_2002_178_consolidated_2026-01-01.htm...,EU_2002_178_consolidated_2026-01-01.html,Article 20 — Responsibilities for feed: feed b...,Article 20
9,Q6,Q6_S2,EU regulatory assessment for a shipment of dri...,5,legal::EU_2002_178_consolidated_2026-01-01.htm...,EU_2002_178_consolidated_2026-01-01.html,Article 20 — Responsibilities for feed: feed b...,Article 20


## 18. Final Build02 diagnostic classification

Interpretation:

- `retrieval_ok_dense`: original dense Top-5 already had the support.
- `improved_by_single_query_hybrid`: hybrid fixed the miss without decomposition.
- `improved_by_multiquery`: decomposition recovered it.
- `topk_capacity_issue`: evidence appears at multi-query Top-10 but not Top-5.
- `still_missed`: evidence exists in the library but even multi-query Top-10 misses it.
- `source_or_chunk_failure`: evidence is absent from the library.


In [19]:
final_diag_rows = []

for qid in [f"Q{i}" for i in range(1, 7)]:
    for group_name, alternatives in SUPPORT_GROUPS[qid].items():

        present = any(
            group_matches(c, alternatives)
            for c in legal_chunks
        )

        dense5 = any(
            group_matches(legal_chunks[i], alternatives)
            for i in top_indices(qid, "dense_baseline", 5)
        )

        hybrid5 = any(
            group_matches(legal_chunks[i], alternatives)
            for i in top_indices(qid, "hybrid_rrf_diverse", 5)
        )

        mq5 = any(
            group_matches(legal_chunks[i], alternatives)
            for i in multiquery_topk(qid, 5)
        )

        mq10 = any(
            group_matches(legal_chunks[i], alternatives)
            for i in multiquery_topk(qid, 10)
        )

        if not present:
            diagnosis = "source_or_chunk_failure"
        elif dense5:
            diagnosis = "retrieval_ok_dense"
        elif hybrid5:
            diagnosis = "improved_by_single_query_hybrid"
        elif mq5:
            diagnosis = "improved_by_multiquery"
        elif mq10:
            diagnosis = "topk_capacity_issue"
        else:
            diagnosis = "still_missed"

        final_diag_rows.append({
            "qid": qid,
            "support_group": group_name,
            "present_in_library": present,
            "dense_top5": dense5,
            "single_hybrid_top5": hybrid5,
            "multiquery_top5": mq5,
            "multiquery_top10": mq10,
            "diagnosis": diagnosis,
        })

final_diag_df = pd.DataFrame(final_diag_rows)

display(final_diag_df)

print("\nFinal diagnosis counts:")
display(
    final_diag_df["diagnosis"]
    .value_counts()
    .rename_axis("diagnosis")
    .reset_index(name="n")
)


,qid,support_group,present_in_library,dense_top5,single_hybrid_top5,multiquery_top5,multiquery_top10,diagnosis
0,Q1,contaminant_limits_2023_915,True,False,True,True,True,improved_by_single_query_hybrid
1,Q1,special_import_2019_1793,True,True,True,True,True,retrieval_ok_dense
2,Q1,general_food_law_178_2002,True,False,False,False,False,still_missed
3,Q1,official_controls_2017_625,True,False,False,False,False,still_missed
4,Q2,dried_figs_6_10_row,True,False,False,True,True,improved_by_multiquery
5,Q3,dried_figs_scope_row,True,False,True,True,True,improved_by_single_query_hybrid
6,Q3,article_3_rule,True,False,True,False,False,improved_by_single_query_hybrid
7,Q4,turkiye_figs_annex_ii,True,True,True,True,True,retrieval_ok_dense
8,Q5,annex_ii_scope_and_30pct,True,False,False,True,True,improved_by_multiquery
9,Q5,identification_code,True,False,False,False,False,still_missed



Final diagnosis counts:


,diagnosis,n
0,still_missed,6
1,improved_by_single_query_hybrid,4
2,retrieval_ok_dense,3
3,improved_by_multiquery,2
4,topk_capacity_issue,1


## 19. Save the final experiment outputs

The candidate retrieval file is **still not automatically treated as final**.

Review Q1–Q6 first, especially Q5/Q6.  
If the method is accepted, apply the same algorithm to Q7–Q10 before freezing the Build02 retrieval version.


In [20]:
MULTIQUERY_COVERAGE_PATH = (
    DATA_DIR / "retrieval_v3_multiquery_coverage_q1_q6.csv"
)
MULTIQUERY_DIAG_PATH = (
    DATA_DIR / "retrieval_v3_multiquery_diagnosis_q1_q6.csv"
)
MULTIQUERY_DEBUG_PATH = (
    DATA_DIR / "retrieval_v3_multiquery_subquery_debug_q1_q6.csv"
)
MULTIQUERY_TOP5_PATH = (
    DATA_DIR / "retrieval_v3_multiquery_top5_q1_q6.csv"
)
MULTIQUERY_DECOMP_PATH = (
    DATA_DIR / "retrieval_v3_query_decomposition_q1_q6.csv"
)

all_coverage_compare.to_csv(
    MULTIQUERY_COVERAGE_PATH,
    index=False,
    encoding="utf-8-sig",
)
final_diag_df.to_csv(
    MULTIQUERY_DIAG_PATH,
    index=False,
    encoding="utf-8-sig",
)
subquery_debug_df.to_csv(
    MULTIQUERY_DEBUG_PATH,
    index=False,
    encoding="utf-8-sig",
)
multi_results_df.to_csv(
    MULTIQUERY_TOP5_PATH,
    index=False,
    encoding="utf-8-sig",
)
decomposition_df.to_csv(
    MULTIQUERY_DECOMP_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:")
for p in [
    MULTIQUERY_COVERAGE_PATH,
    MULTIQUERY_DIAG_PATH,
    MULTIQUERY_DEBUG_PATH,
    MULTIQUERY_TOP5_PATH,
    MULTIQUERY_DECOMP_PATH,
]:
    print(" -", p)


Saved:
 - data\retrieval_v3_multiquery_coverage_q1_q6.csv
 - data\retrieval_v3_multiquery_diagnosis_q1_q6.csv
 - data\retrieval_v3_multiquery_subquery_debug_q1_q6.csv
 - data\retrieval_v3_multiquery_top5_q1_q6.csv
 - data\retrieval_v3_query_decomposition_q1_q6.csv


# Build02 stop rule

After this experiment, **do not keep tuning retrieval indefinitely**.

Use the results to decide:

### If multi-query substantially improves Q5/Q6
Freeze this as the final Build02 retrieval design and apply the same generic method to Q1–Q10.

### If Q5/Q6 still require many missing provisions
Record this as the limit of static RAG:

> A single fixed retrieval stage—even with hybrid and multi-query search—may be insufficient for multi-step regulatory investigation.

That limitation becomes the motivation for:

- **Build03:** tool-assisted search
- **Build04:** iterative agent workflow

This is a useful result, not a failed project.
